# Lab 5 — Workflow Design with LLMs
## Prompt Templates and Output Parsers

**Name:** Suraj Pujari  
**Roll No:** SCFP125013  
**Programme:** MCA  
**Semester:** 3  
**Repository:** `GenAi_LLM_Engineering_SCFP125013`

## Objective

Build six practical LLM workflows using LangChain prompt templates and output parsers. Each step passes its parsed output to the next step instead of repeatedly sending the original raw input.

**Model:** `openai/gpt-oss-20b` through the Groq API.

> Run the notebook from top to bottom. All examples use fictional/demo text. Do not enter private customer, candidate, or health information.


## 1. Install and import dependencies

Run this cell once in the Python environment selected for this notebook. The `%pip` command installs packages into the active Jupyter kernel. Restart the kernel if prompted, then continue from the imports.

Keep your Groq API key in the project-root `.env` file. Never print the key or commit `.env` to GitHub.


In [1]:
%pip install -q langchain-core langchain-groq pydantic python-dotenv requests


Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
from typing import List

import requests
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser


/Users/surajpujari/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/surajpujari/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configure the Groq model

This cell searches the current working directory and its parent directories for `.env`. It reads `GROQ_API_KEY` without displaying it. If your `.env` is in the project root, run the notebook from the project or its subfolder.


In [3]:
from pathlib import Path

def load_project_env():
    current = Path.cwd().resolve()
    for folder in (current, *current.parents):
        env_file = folder / ".env"
        if env_file.is_file():
            load_dotenv(env_file)
            return env_file
    load_dotenv()
    return None

env_path = load_project_env()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError(
        "GROQ_API_KEY was not found. Add GROQ_API_KEY=your_actual_key_here "
        "to your local .env file, then restart the kernel and rerun this cell."
    )

MODEL_NAME = "openai/gpt-oss-20b"
llm = ChatGroq(model=MODEL_NAME, api_key=GROQ_API_KEY, temperature=0.2)
str_parser = StrOutputParser()

print("Groq client is ready.")
print("Model:", MODEL_NAME)
print("Environment file found:", bool(env_path))


Groq client is ready.
Model: openai/gpt-oss-20b
Environment file found: True


In [4]:
def print_section(title, value):
    print(f"\\n{'=' * 12} {title} {'=' * 12}")
    if isinstance(value, BaseModel):
        print(value.model_dump_json(indent=2))
    else:
        print(value)


---
# Exercise 1 — Customer Review to Support Ticket Pipeline

**Input:** `review_text`

- **Step 1:** Extract complaint, product/feature, and sentiment using `PromptTemplate` + `PydanticOutputParser`.
- **Step 2:** Create an internal support ticket using `ChatPromptTemplate` + `StrOutputParser`, based only on the structured Step 1 output.

**Saved-output note:** The saved Exercise 1 result extracts `monthly plan` as the product/feature. Rerun Exercise 1 cells 8 and 9 only if its prompt or input changes.

In [5]:
class ReviewDetails(BaseModel):
    complaint: str = Field(description="The main issue reported by the customer")
    product_or_feature: str = Field(description="Product or feature mentioned; use 'Not specified' if absent")
    sentiment: str = Field(description="Customer sentiment: Positive, Negative, or Mixed")

review_parser = PydanticOutputParser(pydantic_object=ReviewDetails)
review_extract_prompt = PromptTemplate(
    template=(
        "Extract the key details from this customer review. Use only facts in the review. "
        "Choose sentiment from Positive, Negative, or Mixed. Treat a named product, subscription plan, "
        "plan tier, page, or feature as the product_or_feature; for example, extract 'monthly plan' when it appears. "
        "Write 'Not specified' only when no product or feature is mentioned.\\n\\n"
        "Review:\\n{review_text}\\n\\n{format_instructions}"
    ),
    input_variables=["review_text"],
    partial_variables={"format_instructions": review_parser.get_format_instructions()},
)
ticket_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a support ticket writer. Write a concise internal ticket summary using only the structured details provided. Do not invent account details, dates, or troubleshooting steps."),
    ("human", "Create a support ticket summary from these extracted details:\\n{review_details}")
])
review_step_1 = review_extract_prompt | llm | review_parser
review_step_2 = ticket_prompt | llm | str_parser

def customer_review_pipeline(review_text):
    extracted = review_step_1.invoke({"review_text": review_text})
    ticket = review_step_2.invoke({"review_details": extracted.model_dump_json(indent=2)})
    print_section("Step 1 — Structured complaint", extracted)
    print_section("Step 2 — Support ticket summary", ticket)
    return {"structured_complaint": extracted, "support_ticket": ticket}


In [6]:
review_text = (
    "I've been charged twice for the same monthly plan. The billing page shows two payments, "
    "and I cannot find a way to request a refund. The rest of the app works fine."
)
exercise_1_result = customer_review_pipeline(review_text)


\n============ Step 1 — Structured complaint ============
{
  "complaint": "Charged twice for the same monthly plan and unable to request a refund via the billing page.",
  "product_or_feature": "monthly plan",
  "sentiment": "Negative"
}
\n============ Step 2 — Support ticket summary ============
**Ticket Summary**

- **Product/Feature:** Monthly plan  
- **Issue:** Customer was charged twice for the same monthly plan and is unable to request a refund through the billing page.  
- **Sentiment:** Negative  


---
# Exercise 2 — Research Paper Abstract to Layperson Summary Pipeline

**Input:** `abstract_text`

- **Step 1:** Extract the research question, method, and key finding using `PromptTemplate` + `PydanticOutputParser`.
- **Step 2:** Explain only the structured extraction in plain language using `ChatPromptTemplate` + `StrOutputParser`.


In [7]:
class ResearchDetails(BaseModel):
    research_question: str = Field(description="The main question the study investigates")
    method: str = Field(description="The method or approach described in the abstract")
    key_finding: str = Field(description="The main finding; preserve uncertainty and limitations")

research_parser = PydanticOutputParser(pydantic_object=ResearchDetails)
research_extract_prompt = PromptTemplate(
    template=(
        "Extract the research question, method, and key finding from this abstract. "
        "Do not add claims that are not in the abstract. If a detail is missing, say 'Not stated'.\\n\\n"
        "Abstract:\\n{abstract_text}\\n\\n{format_instructions}"
    ),
    input_variables=["abstract_text"],
    partial_variables={"format_instructions": research_parser.get_format_instructions()},
)
layperson_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a science communicator explaining research to a general audience. Use clear everyday language, define technical terms briefly, and do not overstate the finding. If the extraction names a 'spaced-practice group' but only lists daily and weekly schedules without explicitly mapping that label, say only that the group described as using spaced practice recalled more words. Do not say that either the daily or weekly group recalled more words."),
    ("human", "Write a short plain-language summary using only these extracted details. Factual constraint: the extracted method does not explicitly map the phrase 'spaced-practice group' to either the daily or weekly schedule. Do not say that the daily or weekly group recalled more words. Instead, say only that the group described as using spaced practice recalled more words.\\n{research_details}")
])
research_step_1 = research_extract_prompt | llm | research_parser
research_step_2 = layperson_prompt | llm | str_parser

def research_summary_pipeline(abstract_text):
    extracted = research_step_1.invoke({"abstract_text": abstract_text})
    summary = research_step_2.invoke({"research_details": extracted.model_dump_json(indent=2)})
    print_section("Step 1 — Structured extraction", extracted)
    print_section("Step 2 — Layperson summary", summary)
    return {"extraction": extracted, "summary": summary}


In [5]:
abstract_text = (
    "This study examined whether spaced practice improves vocabulary retention among adult language learners. "
    "Sixty learners were assigned to either daily short review sessions or one longer weekly session over four weeks. "
    "The spaced-practice group recalled more words on a test one week later. The study was limited to one course "
    "and does not establish whether the effect is the same for all age groups or languages."
)
exercise_2_result = research_summary_pipeline(abstract_text)


\n============ Step 1 — Structured extraction ============
{
  "research_question": "Does spaced practice improve vocabulary retention among adult language learners?",
  "method": "Sixty adult language learners were assigned to either daily short review sessions or one longer weekly session over four weeks, with recall tested one week later.",
  "key_finding": "The spaced-practice group recalled more words on a test one week later. The study was limited to one course and does not establish whether the effect is the same for all age groups or languages."
}
\n============ Step 2 — Layperson summary ============
**Short‑plain‑language summary**

Researchers wanted to know if *spaced practice*—reviewing new words in short, regular bursts instead of one long session—helps adults keep vocabulary in memory.  

In the study, 60 adult language learners were split into two groups. One group did brief review sessions every day, while the other did one longer review session each week, over a four‑

---
# Exercise 3 — Interview Transcript to Candidate Scorecard Pipeline

**Inputs:** `transcript_text`, `skills_to_assess`

- **Step 1:** Extract interview question-and-answer pairs with `PromptTemplate` + `PydanticOutputParser`.
- **Step 2:** Evaluate each requested skill with `ChatPromptTemplate` + `PydanticOutputParser`, noting evidence or a gap.
- **Step 3:** Format a structured scorecard and cautious recommendation with `ChatPromptTemplate` + `PydanticOutputParser`.

This is a demonstration of structured evidence review, not a substitute for a fair, human-led hiring decision. Mark a skill as not demonstrated when the transcript lacks clear evidence.


In [9]:
class InterviewAnswer(BaseModel):
    question: str = Field(description="Interview question or topic")
    answer: str = Field(description="Candidate answer, summarized faithfully")

class InterviewAnswers(BaseModel):
    answers: List[InterviewAnswer] = Field(description="Question and answer pairs extracted from the transcript")

class SkillEvaluation(BaseModel):
    skill: str = Field(description="Skill being assessed")
    demonstrated: bool = Field(description="True only when the transcript contains clear evidence")
    evidence_or_gap: str = Field(description="Specific evidence from the answer or what evidence is missing")

class SkillEvaluations(BaseModel):
    evaluations: List[SkillEvaluation] = Field(description="One evaluation for each requested skill")

class ScorecardRow(BaseModel):
    skill: str = Field(description="Skill assessed")
    result: str = Field(description="Demonstrated or Not clearly demonstrated")
    evidence_or_gap: str = Field(description="Evidence or gap supporting the result")

class CandidateScorecard(BaseModel):
    scorecard: List[ScorecardRow] = Field(description="Structured hiring scorecard rows")
    overall_recommendation: str = Field(description="Evidence-based next step, not a definitive hiring decision")

answers_parser = PydanticOutputParser(pydantic_object=InterviewAnswers)
skills_parser = PydanticOutputParser(pydantic_object=SkillEvaluations)
scorecard_parser = PydanticOutputParser(pydantic_object=CandidateScorecard)

answers_prompt = PromptTemplate(
    template=(
        "Extract interview questions/topics and candidate answers from this transcript. "
        "Keep meaning accurate; do not infer skills that are not stated.\\n\\n"
        "Transcript:\\n{transcript_text}\\n\\n{format_instructions}"
    ),
    input_variables=["transcript_text"],
    partial_variables={"format_instructions": answers_parser.get_format_instructions()},
)
skills_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a technical interviewer evaluating skill evidence. Assess only the supplied answers. Mark demonstrated=true only when clear evidence exists. If evidence is weak or absent, mark false and explain the gap. Return one evaluation per requested skill."),
    ("human", "Skills to assess: {skills_to_assess}\\n\\nExtracted interview answers:\\n{answers_json}\\n\\n{format_instructions}")
]).partial(format_instructions=skills_parser.get_format_instructions())
scorecard_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a hiring panel coordinator. Convert the supplied skill evaluations into a structured scorecard. Keep the recommendation cautious and evidence-based; do not make a final hiring decision."),
    ("human", "Skill evaluations:\\n{evaluations_json}\\n\\n{format_instructions}")
]).partial(format_instructions=scorecard_parser.get_format_instructions())

interview_step_1 = answers_prompt | llm | answers_parser
interview_step_2 = skills_prompt | llm | skills_parser
interview_step_3 = scorecard_prompt | llm | scorecard_parser

def candidate_scorecard_pipeline(transcript_text, skills_to_assess):
    answers = interview_step_1.invoke({"transcript_text": transcript_text})
    evaluations = interview_step_2.invoke({
        "skills_to_assess": ", ".join(skills_to_assess),
        "answers_json": answers.model_dump_json(indent=2),
    })
    scorecard = interview_step_3.invoke({"evaluations_json": evaluations.model_dump_json(indent=2)})
    print_section("Step 1 — Extracted answers", answers)
    print_section("Step 2 — Skill evidence evaluation", evaluations)
    print_section("Step 3 — Final scorecard", scorecard)
    return {"answers": answers, "evaluations": evaluations, "scorecard": scorecard}


In [10]:
transcript_text = (
    "Interviewer: How did you use Java in a project?\\n"
    "Candidate: I built REST endpoints in Spring Boot and used service classes to keep business logic separate.\\n"
    "Interviewer: How did you test the API?\\n"
    "Candidate: I used Postman to check successful requests and invalid input, but I have not written automated integration tests yet.\\n"
    "Interviewer: Tell me about SQL performance.\\n"
    "Candidate: I know basic joins and indexes from coursework, but I have not tuned a production database."
)
skills_to_assess = ["Java and Spring Boot", "API testing", "SQL performance tuning"]
exercise_3_result = candidate_scorecard_pipeline(transcript_text, skills_to_assess)


\n============ Step 1 — Extracted answers ============
{
  "answers": [
    {
      "question": "How did you use Java in a project?",
      "answer": "I built REST endpoints in Spring Boot and used service classes to keep business logic separate."
    },
    {
      "question": "How did you test the API?",
      "answer": "I used Postman to check successful requests and invalid input, but I have not written automated integration tests yet."
    },
    {
      "question": "Tell me about SQL performance.",
      "answer": "I know basic joins and indexes from coursework, but I have not tuned a production database."
    }
  ]
}
\n============ Step 2 — Skill evidence evaluation ============
{
  "evaluations": [
    {
      "skill": "Java and Spring Boot",
      "demonstrated": true,
      "evidence_or_gap": "Built REST endpoints in Spring Boot and used service classes to separate business logic."
    },
    {
      "skill": "API testing",
      "demonstrated": true,
      "evidence_or_gap":

---
# Exercise 4 — Outdoor Event Plan to Weather-Adjusted Recommendation Pipeline

**Input:** `event_description_text`

- **Step 1:** Extract event type, date, location, and indoor/outdoor preference using `PromptTemplate` + `PydanticOutputParser`.
- **Step 2:** Format the weather query with `PromptTemplate`, call the Open-Meteo API, and normalize actual API values into `WeatherDetails` with `PydanticOutputParser`.
- **Step 3:** Generate an organizer recommendation from event and weather data using `ChatPromptTemplate` + `StrOutputParser`.

Open-Meteo's forecast range depends on the current date and service availability. If the date is outside the available forecast window, the notebook reports unavailable data rather than inventing a forecast.

**Saved-output note:** The stored Pune result is an API observation for the event date `2026-10-25`. It is historical relative to a later run and must not be presented as a current forecast. Rerunning the workflow with a supported future date requests a new weather result.


In [11]:
class EventDetails(BaseModel):
    event_type: str = Field(description="Type of event")
    event_date: str = Field(description="Event date in YYYY-MM-DD format; use 'Not specified' if absent")
    location: str = Field(description="City or location name")
    indoor_outdoor_preference: str = Field(description="Indoor, Outdoor, or Flexible")

class WeatherDetails(BaseModel):
    temperature: str = Field(description="Temperature in Celsius, or unavailable")
    precipitation_chance: str = Field(description="Precipitation probability percentage, or unavailable")
    conditions: str = Field(description="Weather condition description or unavailable")

event_parser = PydanticOutputParser(pydantic_object=EventDetails)
weather_parser = PydanticOutputParser(pydantic_object=WeatherDetails)
event_prompt = PromptTemplate(
    template=(
        "Extract event details from the description. Use YYYY-MM-DD for a provided date. "
        "Do not guess a missing date or location; write 'Not specified'. Indoor/outdoor preference "
        "must be Indoor, Outdoor, or Flexible.\\n\\nDescription:\\n{event_description_text}\\n\\n{format_instructions}"
    ),
    input_variables=["event_description_text"],
    partial_variables={"format_instructions": event_parser.get_format_instructions()},
)
weather_query_prompt = PromptTemplate.from_template(
    "Look up weather for location={location} on date={event_date}. Use returned weather-service values only."
)
event_recommendation_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an event planning assistant. Recommend whether to proceed as planned, reschedule, or move indoors. Base the recommendation only on the supplied event details and weather data. Consider high temperatures, precipitation chance, weather conditions, outdoor exposure, and whether an indoor alternative exists. Treat temperatures around 33°C as hot. Mention practical precautions such as shade, water, and breaks when relevant. If weather data is unavailable or incomplete, state the limitation and do not invent a forecast. Do not claim an event is completely safe based on weather data alone."),
    ("human", "Event details:\\n{event_json}\\n\\nWeather data:\\n{weather_json}\\n\\nWrite a concise recommendation to the organizer and explain the reasons and precautions.")
])
event_step_1 = event_prompt | llm | event_parser
event_step_3 = event_recommendation_prompt | llm | str_parser

WEATHER_CODES = {
    0: "Clear sky", 1: "Mainly clear", 2: "Partly cloudy", 3: "Overcast",
    45: "Fog", 48: "Depositing rime fog", 51: "Light drizzle",
    53: "Moderate drizzle", 55: "Dense drizzle", 61: "Slight rain",
    63: "Moderate rain", 65: "Heavy rain", 71: "Slight snow",
    73: "Moderate snow", 75: "Heavy snow", 80: "Rain showers",
    81: "Moderate rain showers", 82: "Violent rain showers",
    95: "Thunderstorm", 96: "Thunderstorm with hail", 99: "Thunderstorm with heavy hail",
}

def fetch_open_meteo_weather(location, event_date):
    geo_response = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": location, "count": 1, "language": "en", "format": "json"},
        timeout=20,
    )
    geo_response.raise_for_status()
    results = geo_response.json().get("results", [])
    if not results:
        raise ValueError(f"Could not find a location for: {location}")
    place = results[0]

    query = weather_query_prompt.invoke({
        "location": f"{place['name']}, {place.get('country', '')}",
        "event_date": event_date,
    })
    print("Weather query:", query.to_string())

    forecast_response = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": place["latitude"],
            "longitude": place["longitude"],
            "daily": "temperature_2m_max,precipitation_probability_max,weather_code",
            "timezone": "auto",
            "start_date": event_date,
            "end_date": event_date,
        },
        timeout=20,
    )
    forecast_response.raise_for_status()
    daily = forecast_response.json().get("daily", {})
    if not daily.get("time"):
        raw_weather = {
            "temperature": "Unavailable for this date",
            "precipitation_chance": "Unavailable for this date",
            "conditions": "No forecast returned by the weather service",
        }
    else:
        temp = daily.get("temperature_2m_max", [None])[0]
        precip = daily.get("precipitation_probability_max", [None])[0]
        code_value = daily.get("weather_code", [None])[0]
        raw_weather = {
            "temperature": f"{temp} °C" if temp is not None else "Unavailable",
            "precipitation_chance": f"{precip}%" if precip is not None else "Unavailable",
            "conditions": WEATHER_CODES.get(
                code_value,
                f"Weather code {code_value}" if code_value is not None else "Unavailable",
            ),
        }
    return weather_parser.parse(json.dumps(raw_weather))

def event_weather_pipeline(event_description_text):
    event = event_step_1.invoke({"event_description_text": event_description_text})
    if event.event_date == "Not specified" or event.location == "Not specified":
        raise ValueError("Provide a specific event date and location before requesting weather.")
    weather = fetch_open_meteo_weather(event.location, event.event_date)
    recommendation = event_step_3.invoke({
        "event_json": event.model_dump_json(indent=2),
        "weather_json": weather.model_dump_json(indent=2),
    })
    print_section("Step 1 — Event details", event)
    print_section("Step 2 — Weather data", weather)
    print_section("Step 3 — Organizer recommendation", recommendation)
    return {"event": event, "weather": weather, "recommendation": recommendation}


In [12]:
event_description_text = (
    "We are planning an outdoor community book fair in Pune on 2026-10-25. "
    "The stalls and children's reading area will be outside, but we can move the main activities indoors if needed."
)
try:
    exercise_4_result = event_weather_pipeline(event_description_text)
except (requests.RequestException, ValueError, KeyError) as error:
    print("Weather pipeline could not complete:", error)
    print("Check the event date/location or try a date supported by the forecast API.")


Weather query: Look up weather for location=Pune, India on date=2026-10-25. Use returned weather-service values only.
\n============ Step 1 — Event details ============
{
  "event_type": "community book fair",
  "event_date": "2026-10-25",
  "location": "Pune",
  "indoor_outdoor_preference": "Flexible"
}
\n============ Step 2 — Weather data ============
{
  "temperature": "33.3 °C",
  "precipitation_chance": "6%",
  "conditions": "Overcast"
}
\n============ Step 3 — Organizer recommendation ============
**Recommendation:** Proceed outdoors with precautions (do not reschedule or move indoors unless conditions change).

**Why:**  
- **Temperature:** 33.3 °C is hot enough to cause heat‑related discomfort, but not extreme.  
- **Precipitation:** 6 % chance of rain is very low, so flooding or sudden showers are unlikely.  
- **Conditions:** Overcast reduces direct sun, but the ambient heat remains.  

**Practical precautions:**  
1. **Shade structures** (tents, awnings, umbrellas) at key ac

---
# Exercise 5 — Customer Support Chat Log to Escalation Report Pipeline

**Input:** `chat_log_text`

- **Step 1:** Normalize the chat into issue, customer tone, and resolution attempted using `PromptTemplate` + `PydanticOutputParser`.
- **Step 2:** Check escalation readiness and missing information using `ChatPromptTemplate` + `PydanticOutputParser`.
- **Step 3:** Write a prioritized escalation report using `ChatPromptTemplate` + `StrOutputParser`, based only on the structured summary and gap check.


In [13]:
class SupportSummary(BaseModel):
    issue: str = Field(description="Main problem reported by the customer")
    customer_tone: str = Field(description="Customer tone, based on the text")
    resolution_attempted: str = Field(description="Troubleshooting already attempted; say 'Not stated' if absent")

class EscalationReadiness(BaseModel):
    missing_fields: List[str] = Field(description="Important missing details such as account ID, error code, or timestamps")
    is_ready_to_escalate: bool = Field(description="True only if enough information exists for a useful handoff")

support_parser = PydanticOutputParser(pydantic_object=SupportSummary)
readiness_parser = PydanticOutputParser(pydantic_object=EscalationReadiness)
support_extract_prompt = PromptTemplate(
    template=(
        "Normalize this support chat into issue, customer tone, and resolution attempted. "
        "Do not invent missing facts.\\n\\nChat log:\\n{chat_log_text}\\n\\n{format_instructions}"
    ),
    input_variables=["chat_log_text"],
    partial_variables={"format_instructions": support_parser.get_format_instructions()},
)
readiness_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an escalation-readiness reviewer. Check whether the structured summary has enough information for a useful handoff. Consider account ID, error code, affected service, and relevant time when applicable. Do not invent missing values; list gaps explicitly."),
    ("human", "Structured support summary:\\n{summary_json}\\n\\n{format_instructions}")
]).partial(format_instructions=readiness_parser.get_format_instructions())
escalation_report_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a support operations lead. Write a concise escalation report using only the structured summary and readiness review. Include priority, reason, issue, resolution attempted, missing information, and next action. P1 means critical widespread outage; P2 major service issue; P3 standard issue with limited or unclear impact; P4 minor issue/request. Do not assign P1 without evidence of critical impact. If information is insufficient, state that priority is provisional. Never invent details."),
    ("human", "Support summary:\\n{summary_json}\\n\\nReadiness review:\\n{readiness_json}\\n\\nPrepare the escalation report.")
])
support_step_1 = support_extract_prompt | llm | support_parser
support_step_2 = readiness_prompt | llm | readiness_parser
support_step_3 = escalation_report_prompt | llm | str_parser

def escalation_pipeline(chat_log_text):
    summary = support_step_1.invoke({"chat_log_text": chat_log_text})
    readiness = support_step_2.invoke({"summary_json": summary.model_dump_json(indent=2)})
    report = support_step_3.invoke({
        "summary_json": summary.model_dump_json(indent=2),
        "readiness_json": readiness.model_dump_json(indent=2),
    })
    print_section("Step 1 — Structured support summary", summary)
    print_section("Step 2 — Escalation readiness", readiness)
    print_section("Step 3 — Escalation report", report)
    return {"summary": summary, "readiness": readiness, "report": report}


In [14]:
chat_log_text = (
    "Customer: The checkout keeps failing and I have tried again three times.\\n"
    "Agent: Please clear your browser cache and try once more.\\n"
    "Customer: I did that, but it still says 'payment could not be processed'. "
    "I need to place the order today.\\n"
    "Agent: I will pass this to the technical team.\\n"
    "Note: No account ID, transaction reference, or exact error code was recorded."
)
exercise_5_result = escalation_pipeline(chat_log_text)


\n============ Step 1 — Structured support summary ============
{
  "issue": "Checkout process fails with payment error",
  "customer_tone": "Frustrated and urgent",
  "resolution_attempted": "Cleared browser cache; escalated to technical team"
}
\n============ Step 2 — Escalation readiness ============
{
  "missing_fields": [
    "account ID",
    "error code",
    "affected service",
    "timestamp"
  ],
  "is_ready_to_escalate": false
}
\n============ Step 3 — Escalation report ============
**Escalation Report**

- **Priority:** P3 – Standard issue with limited or unclear impact (priority provisional due to missing critical data).  
- **Reason:** Checkout process fails with a payment error; customer is frustrated and urgent.  
- **Issue:** Payment failure during checkout.  
- **Resolution Attempted:** Cleared browser cache; escalated to technical team.  
- **Missing Information:**  
  - Account ID  
  - Error code  
  - Affected service  
  - Timestamp  
- **Next Action:**  
  1. Ob

---
# Exercise 6 — Workout Goal to Weekly Plan Pipeline

**Inputs:** `fitness_goal_text`, `days_per_week`

- **Step 1:** Extract goal type, fitness level, available days, and equipment access using `PromptTemplate` + `PydanticOutputParser`.
- **Step 2:** Generate a structured day-by-day weekly plan using `ChatPromptTemplate` + `PydanticOutputParser`, based only on the extracted details.

The sample is general fitness information, not medical advice. Anyone with an injury or medical concern should consult a qualified professional before starting a new exercise plan.


In [15]:
class FitnessGoalDetails(BaseModel):
    goal_type: str = Field(description="Main fitness goal")
    fitness_level: str = Field(description="Current fitness level as described")
    days_available: int = Field(description="Number of available workout days per week")
    equipment_access: str = Field(description="Equipment access; say 'Not specified' if absent")

class WorkoutDay(BaseModel):
    day: str = Field(description="Day name or workout day number")
    focus: str = Field(description="Workout focus")
    activities: List[str] = Field(description="Exercises or activities for this day")
    duration_minutes: int = Field(ge=0, le=180, description="Approximate workout duration; use 0 for rest")
    notes: str = Field(description="Rest, warm-up, or safety note")

class WeeklyWorkoutPlan(BaseModel):
    plan: List[WorkoutDay] = Field(description="Day-by-day plan entries; include rest days as needed")

fitness_parser = PydanticOutputParser(pydantic_object=FitnessGoalDetails)
workout_parser = PydanticOutputParser(pydantic_object=WeeklyWorkoutPlan)
fitness_extract_prompt = PromptTemplate(
    template=(
        "Extract the person's fitness goal, current level, available days, and equipment access. "
        "Use the supplied days_per_week value for days_available. Do not infer medical conditions. "
        "If equipment access is not mentioned, write 'Not specified'.\\n\\n"
        "Goal description:\\n{fitness_goal_text}\\nDays available per week: {days_per_week}\\n\\n{format_instructions}"
    ),
    input_variables=["fitness_goal_text", "days_per_week"],
    partial_variables={"format_instructions": fitness_parser.get_format_instructions()},
)
workout_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a certified personal trainer. Create a realistic beginner-appropriate weekly plan using only the extracted details. Schedule workouts on the available days and include rest days as appropriate. Include warm-up and cool-down guidance. Do not assume unavailable equipment. duration_minutes must be an integer from 0 to 180; use 0 for rest. Return valid structured output."),
    ("human", "Extracted goal details:\\n{goal_details_json}\\n\\n{format_instructions}")
]).partial(format_instructions=workout_parser.get_format_instructions())
fitness_step_1 = fitness_extract_prompt | llm | fitness_parser
fitness_step_2 = workout_prompt | llm | workout_parser

def workout_pipeline(fitness_goal_text, days_per_week):
    if not isinstance(days_per_week, int) or not 1 <= days_per_week <= 7:
        raise ValueError("days_per_week must be an integer from 1 to 7.")
    goal = fitness_step_1.invoke({
        "fitness_goal_text": fitness_goal_text,
        "days_per_week": days_per_week,
    })
    plan = fitness_step_2.invoke({"goal_details_json": goal.model_dump_json(indent=2)})
    print_section("Step 1 — Extracted fitness goal", goal)
    print_section("Step 2 — Weekly workout plan", plan)
    return {"goal": goal, "plan": plan}


In [16]:
fitness_goal_text = (
    "I am new to regular exercise and want to improve general fitness and stamina. "
    "I can exercise on weekdays three times each week. I do not have gym equipment, "
    "so I prefer walking and bodyweight exercises."
)
exercise_6_result = workout_pipeline(fitness_goal_text, days_per_week=3)


\n============ Step 1 — Extracted fitness goal ============
{
  "goal_type": "General fitness and stamina",
  "fitness_level": "Beginner",
  "days_available": 3,
  "equipment_access": "No gym equipment"
}
\n============ Step 2 — Weekly workout plan ============
{
  "plan": [
    {
      "day": "Day 1",
      "focus": "Cardio & Core",
      "activities": [
        "Jumping jacks",
        "High knees",
        "Butt kicks",
        "Mountain climbers",
        "Plank"
      ],
      "duration_minutes": 30,
      "notes": "Warm-up: 5 min dynamic stretches (arm circles, leg swings). Cool-down: 5 min static stretches (hamstring, quad, chest)."
    },
    {
      "day": "Day 2",
      "focus": "Strength & Mobility",
      "activities": [
        "Bodyweight squats",
        "Lunges",
        "Push-ups (knees)",
        "Glute bridges",
        "Bird-dog"
      ],
      "duration_minutes": 30,
      "notes": "Warm-up: 5 min dynamic (hip circles, shoulder rolls). Cool-down: 5 min static (hip 

In [17]:
plan = exercise_6_result["plan"]
print("Total plan entries:", len(plan.plan))
for item in plan.plan:
    print(f"{item.day}: {item.focus} | {item.duration_minutes} minutes")


Total plan entries: 7
Day 1: Cardio & Core | 30 minutes
Day 2: Strength & Mobility | 30 minutes
Day 3: Full Body Conditioning | 30 minutes
Day 4: Rest | 0 minutes
Day 5: Rest | 0 minutes
Day 6: Rest | 0 minutes
Day 7: Rest | 0 minutes


## Final checklist

- [ ] Run all cells from top to bottom in the same Python environment.
- [ ] Confirm all six exercises produce outputs without errors.
- [ ] Review the weather API result; a forecast may be unavailable for the chosen date.
- [ ] Save the notebook with outputs generated by your own run.
- [ ] Ensure `.env` is ignored by Git and never commit API credentials.
- [ ] Push the completed lab to your instructor-required repository: `generative_ai_SCFP125013`.

This notebook contains saved outputs from a prior execution. After changing a prompt or workflow, manually rerun the relevant cells before treating their saved output as evidence for the change. The unchecked checklist items require manual confirmation and are not marked complete automatically.
